# Notebook 2: Extracción y Preprocesamiento - Dr. Stephen Stahl (Prescriber's Guide)

Este notebook procesa las fichas técnicas y monografías de la **"Guía del Prescriptor" de Stephen Stahl**.

### Objetivos:
1. Extraer los campos estandarizados de las monografías de Stahl (Clase, Mecanismo receptor, Perfil de citocromos CYP450, Efectos frecuentes vs graves, Perlas clínicas).
2. Modelar con precisión el **perfil de enzimas hepáticas (CYP450)**: fundamental para calcular interacciones farmacocinéticas en polifarmacia.
3. Clasificar los eventos adversos por nivel de severidad (comunes vs potencialmente letales).
4. Exportar el catálogo limpio a CSV para la base de datos relacional SQLite.


In [ ]:
import os
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f'Entorno configurado correctamente. Raíz del proyecto: {project_root}')


## 1. Carga de Datos y Estructura Monográfica de Stahl

La Guía del Prescriptor de Stahl tiene un formato sistemático por fármaco. Definimos un lector con un corpus representativo de psicofármacos.


In [ ]:
CORPUS_MUESTRA_STAHL = """
[DRUG_START]
DRUG: Clozapina
CLASS: Antipsicótico Atípico (Segunda Generación)
RECEPTOR_PROFILE: Antagonista 5-HT2A > D2; alta afinidad por receptores D4, 5-HT2C, alfa-1 adrenérgicos, H1 histaminérgicos y M1 colinérgicos.
CYP_METABOLISM: Sustrato principal de CYP1A2. Sustrato secundario de CYP2D6 y CYP3A4. Los fumadores inducen CYP1A2 y reducen sus niveles.
COMMON_SIDE_EFFECTS: Sedación intensa, sialorrea nocturna, aumento de peso, estreñimiento severo, taquicardia sinusal.
SEVERE_SIDE_EFFECTS: Agranulocitosis (neutropenia grave potencialmente mortal), miocarditis, convulsiones dosis-dependiente, íleo paralítico.
CLINICAL_PEARLS: Indicado en esquizofrenia resistente y riesgo suicida recurrente. Requiere monitorización hematológica estricta con hemogramas seriados.
[DRUG_END]

[DRUG_START]
DRUG: Paroxetina
CLASS: Inhibidor Selectivo de la Recaptación de Serotonina (ISRS)
RECEPTOR_PROFILE: Inhibidor potente del transportador de serotonina (SERT); ligera acción anticolinérgica muscarínica (M1) y débil inhibición de recaptación de noradrenalina (NET).
CYP_METABOLISM: Potente inhibidor del citocromo CYP2D6. Sustrato de CYP2D6.
COMMON_SIDE_EFFECTS: Disfunción sexual marcada, somnolencia/sedación, aumento de peso moderado, náuseas, boca seca por efecto anticolinérgico.
SEVERE_SIDE_EFFECTS: Síndrome de discontinuación intenso tras suspensión brusca, síndrome serotoninérgico, teratogenicidad (defectos septales cardíacos en primer trimestre).
CLINICAL_PEARLS: Es el ISRS más sedante y anticolinérgico. Desaconsejado en adultos mayores por riesgo de deterioro cognitivo y caídas.
[DRUG_END]

[DRUG_START]
DRUG: Bupropión
CLASS: Inhibidor de la Recaptación de Noradrenalina y Dopamina (IRND)
RECEPTOR_PROFILE: Bloqueo débil del transportador de dopamina (DAT) y noradrenalina (NET). Antagonista nicotínico.
CYP_METABOLISM: Metabolizado por CYP2B6. Potente inhibidor de CYP2D6.
COMMON_SIDE_EFFECTS: Insomnio, boca seca, cefalea, temblor, agitación, diaforesis, pérdida de peso.
SEVERE_SIDE_EFFECTS: Disminución del umbral convulsivo (crisis epilépticas en dosis altas o bulimia/anorexia).
CLINICAL_PEARLS: Excelente alternativa para evitar la disfunción sexual y el aumento de peso de los ISRS. Útil en cesación tabáquica y TDAH comórbido.
[DRUG_END]

[DRUG_START]
DRUG: Lamotrigina
CLASS: Estabilizador del Ánimo / Antiepiléptico
RECEPTOR_PROFILE: Bloqueo de canales de sodio voltaje-dependientes; modulación de la liberación presináptica de glutamato.
CYP_METABOLISM: Metabolismo por glucuronidación (UGT1A4). No depende fuertemente de citocromos CYP.
COMMON_SIDE_EFFECTS: Cefalea, mareos, visión borrosa, náuseas, exantema cutáneo benigno.
SEVERE_SIDE_EFFECTS: Síndrome de Stevens-Johnson / Necrólisis Epidérmica Tóxica (exantema grave potencialmente mortal).
CLINICAL_PEARLS: Eficaz principalmente en la fase depresiva del Trastorno Bipolar. El ácido valproico duplica sus niveles (requiere reducir la dosis a la mitad); la carbamazepina reduce sus niveles a la mitad.
[DRUG_END]
"""

print(f"Muestra de monografías Stahl cargada ({len(CORPUS_MUESTRA_STAHL)} caracteres).")


## 2. Segmentación de Monografías y Parser

Parseamos cada ficha monográfica de Stahl delimitada entre `[DRUG_START]` y `[DRUG_END]`.


In [ ]:
def parse_monografias_stahl(texto):
    """Parsea monografías completas de Stahl a DataFrame estructurado."""
    patron = re.compile(
        r"\[DRUG_START\]\s*"
        r"DRUG:\s*(?P<farmaco>[^\n]+)\n"
        r"CLASS:\s*(?P<clase>[^\n]+)\n"
        r"RECEPTOR_PROFILE:\s*(?P<perfil_receptores>[^\n]+)\n"
        r"CYP_METABOLISM:\s*(?P<metabolismo_cyp>[^\n]+)\n"
        r"COMMON_SIDE_EFFECTS:\s*(?P<efectos_comunes>[^\n]+)\n"
        r"SEVERE_SIDE_EFFECTS:\s*(?P<efectos_graves>[^\n]+)\n"
        r"CLINICAL_PEARLS:\s*(?P<perlas_clinicas>[^\n]+)\s*"
        r"\[DRUG_END\]",
        re.MULTILINE
    )
    
    items = []
    for m in patron.finditer(texto):
        d = m.groupdict()
        items.append({k: v.strip() for k, v in d.items()})
        
    return pd.DataFrame(items)

df_stahl_raw = parse_monografias_stahl(CORPUS_MUESTRA_STAHL)
df_stahl_raw.head()


## 3. Extracción de Features Farmacocinéticas: Citocromos CYP450

En Stahl, el comportamiento frente al citocromo P450 es crucial. Creamos flags booleanos/categóricos para identificar si el fármaco inhibe enzimas como **CYP2D6, CYP1A2 o CYP3A4**.


In [ ]:
def analizar_perfil_cyp(texto_cyp):
    """Extrae de forma determinista la interacción enzimática con citocromos."""
    texto = texto_cyp.lower()
    return {
        "inhibe_cyp2d6": 1 if ("inhibidor de cyp2d6" in texto or "inhibe cyp2d6" in texto or "inhibidor potente de cyp2d6" in texto) else 0,
        "inhibe_cyp3a4": 1 if ("inhibidor de cyp3a4" in texto or "inhibe cyp3a4" in texto) else 0,
        "sustrato_cyp1a2": 1 if ("sustrato" in texto and "cyp1a2" in texto) else 0,
        "sustrato_cyp2d6": 1 if ("sustrato" in texto and "cyp2d6" in texto) else 0,
        "sustrato_cyp3a4": 1 if ("sustrato" in texto and "cyp3a4" in texto) else 0,
    }

cyp_features = df_stahl_raw["metabolismo_cyp"].apply(analizar_perfil_cyp).apply(pd.Series)
df_stahl = pd.concat([df_stahl_raw, cyp_features], axis=1)

# Asignar código ATC
ATC_STAHL_MAP = {
    "Clozapina": "N05AH02",
    "Paroxetina": "N06AB05",
    "Bupropión": "N06AX12",
    "Lamotrigina": "N03AX09"
}
df_stahl["atc_codigo"] = df_stahl["farmaco"].map(ATC_STAHL_MAP)
df_stahl[["farmaco", "atc_codigo", "inhibe_cyp2d6", "sustrato_cyp1a2", "sustrato_cyp2d6"]]


## 4. Desagregación de Efectos Adversos por Severidad

Stahl distingue claramente entre efectos secundarios tolerables y eventos adversos con riesgo de muerte. Creamos la tabla normalizada con el atributo `severidad`.


In [ ]:
eventos_stahl = []
for _, row in df_stahl.iterrows():
    # Efectos comunes
    for ef in [e.strip() for e in re.split(r"[,;]", row["efectos_comunes"]) if e.strip()]:
        eventos_stahl.append({
            "farmaco": row["farmaco"],
            "atc_codigo": row["atc_codigo"],
            "efecto_adverso": ef,
            "severidad": "Común / Moderado",
            "fuente": "Stahl - Prescriber's Guide"
        })
    # Efectos graves
    for ef in [e.strip() for e in re.split(r"[,;]", row["efectos_graves"]) if e.strip()]:
        eventos_stahl.append({
            "farmaco": row["farmaco"],
            "atc_codigo": row["atc_codigo"],
            "efecto_adverso": ef,
            "severidad": "Grave / Riesgo Vital",
            "fuente": "Stahl - Prescriber's Guide"
        })

df_eventos_stahl = pd.DataFrame(eventos_stahl)
print(f"Total eventos clasificados: {len(df_eventos_stahl)}")
df_eventos_stahl.head(8)


## 5. Exportación de Resultados

Guardamos los datasets limpios para integración con SQLite.


In [ ]:
out_farmacos = DATA_PROCESSED / "stahl_farmacos_clean.csv"
out_eventos = DATA_PROCESSED / "stahl_eventos_adversos_clean.csv"

df_stahl.to_csv(out_farmacos, index=False, encoding="utf-8")
df_eventos_stahl.to_csv(out_eventos, index=False, encoding="utf-8")

print(f"Fármacos de Stahl guardados en: {out_farmacos}")
print(f"Eventos adversos de Stahl guardados en: {out_eventos}")
